In [ ]:
import os

# Le token est injecté par le CLI Colab via --env HF_TOKEN=$HF_TOKEN
HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "❌ HF_TOKEN non trouvé dans os.environ.\n"
        "Lancez le notebook avec :\n"
        "  colab exec -s trainer -f notebook.ipynb "
        "--env HF_TOKEN=$HF_TOKEN --timeout 3600"
    )

print(f"✅ HF_TOKEN chargé : {HF_TOKEN[:6]}...{HF_TOKEN[-4:]}")

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/llm-dataset")

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)

print(f"✅ CWD : {os.getcwd()}")

# PYTHONPATH
PROJECT_ROOT = PROJECT_DIR
SRC_DIR = PROJECT_ROOT / "src"

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["SRC_DIR"] = str(SRC_DIR)

existing = os.environ.get("PYTHONPATH", "")
new_path = f"{SRC_DIR}:{PROJECT_ROOT}"

if existing:
    new_path = f"{new_path}:{existing}"

os.environ["PYTHONPATH"] = new_path

print(f"   PYTHONPATH : {os.environ['PYTHONPATH'][:80]}...")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Synobotix/llm-dataset.git"
REPO_NAME = "llm-dataset"

PROJECT_DIR = Path(f"/content/{REPO_NAME}")

if PROJECT_DIR.exists():
    print("📁 Repo déjà cloné — pull")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    print("📥 Clonage...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"✅ Projet : {os.getcwd()}")

In [ ]:
import os
import subprocess

print("📦 Installation de Poetry...")

subprocess.run(
    "curl -sSL https://install.python-poetry.org | python3 -",
    shell=True,
    check=True,
)

os.environ["PATH"] = (
    f"{os.path.expanduser('~')}/.local/bin:"
    + os.environ["PATH"]
)

result = subprocess.run(
    ["poetry", "--version"],
    capture_output=True,
    text=True,
)
print(result.stdout)

In [ ]:
import subprocess

print("📦 Installation de Python 3.11...")

subprocess.run(
    ["apt-get", "update"],
    check=True,
)

subprocess.run(
    ["apt-get", "install", "-y",
     "python3.11", "python3.11-venv", "python3.11-dev"],
    check=True,
)

result = subprocess.run(
    ["python3.11", "--version"],
    capture_output=True,
    text=True,
)
print(f"✅ Python 3.11 : {result.stdout.strip()}")

In [ ]:
import subprocess

# ============================================================
# 1. Configurer Poetry pour utiliser Python 3.11
# ============================================================

print("🔧 Configuration de Poetry pour Python 3.11...")

subprocess.run(
    ["poetry", "env", "use", "python3.11"],
    check=True,
)

# ============================================================
# 2. Installer les dépendances
# ============================================================

print("\n📦 Installation des dépendances Poetry...")

subprocess.run(
    ["poetry", "install", "--no-root"],
    check=True,
)

result = subprocess.run(
    ["poetry", "run", "python", "--version"],
    capture_output=True,
    text=True,
)
print(f"✅ Python dans Poetry : {result.stdout.strip()}")

# ============================================================
# 3. Désinstaller torch CPU
# ============================================================

subprocess.run(
    ["poetry", "run", "pip", "uninstall", "-y",
     "torch", "torchvision", "torchaudio"],
    check=False,
)

# ============================================================
# 4. Installer torch CUDA 12.1
# ============================================================

print("\n📦 Installation torch CUDA 12.1...")

result = subprocess.run(
    ["poetry", "run", "pip", "install",
     "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("❌ Échec CUDA")
    print("STDOUT:", result.stdout[-1000:])
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Installation torch CUDA échouée")

print("✅ torch CUDA installé")

# ============================================================
# 5. Vérification finale
# ============================================================

result = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import torch; "
     "print('torch', torch.__version__); "
     "print('cuda', torch.cuda.is_available()); "
     "print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"],
    capture_output=True,
    text=True,
)

print("\n" + "=" * 70)
print("RÉSULTAT FINAL")
print("=" * 70)
print(result.stdout)

In [ ]:
import os
from huggingface_hub import login, whoami

login(
    token=os.environ["HF_TOKEN"],
    add_to_git_credential=False,
)

info = whoami(token=os.environ["HF_TOKEN"])
print(f"✅ HF connecté : {info['name']}")

In [ ]:
import os
from pathlib import Path

# Nettoyer les fichiers résiduels (au cas où la session aurait déjà tourné)
folders_to_clean = [
    "data/processed",
    "data/tokenized",
    "tokenizer",
    "checkpoints/bitnet",
    "documentation/result/bitnet",
]

for folder in folders_to_clean:
    path = Path(folder)
    if path.exists():
        for f in path.glob("*"):
            if f.is_file():
                f.unlink()
                print(f"🗑️ Supprimé : {f}")
    else:
        path.mkdir(parents=True, exist_ok=True)

print("\n✅ Nettoyage terminé")

In [ ]:
import os
import subprocess

print("=" * 70)
print("ÉTAPE 9 — TÉLÉCHARGEMENT NEMOTRON")
print("=" * 70)

result = subprocess.run(
    ["poetry", "run", "python", "-m",
     "scripts.nemotron.download_and_prepare"],
    env=os.environ.copy(),
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("❌ Nemotron échoué")

print("\n✅ Nemotron terminé")

In [ ]:
import os
import subprocess

print("=" * 70)
print("ÉTAPE 10 — SPLIT TRAIN/VALIDATION")
print("=" * 70)

result = subprocess.run(
    ["poetry", "run", "python", "-m",
     "scripts.split_dataset"],
    env=os.environ.copy(),
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("❌ Split échoué")

print("\n✅ Split terminé")

In [ ]:
import os
import subprocess

print("=" * 70)
print("ÉTAPE 11 — ENTRAÎNEMENT DU TOKENIZER")
print("=" * 70)

result = subprocess.run(
    ["poetry", "run", "python", "-m",
     "tokenizer.train_tokenizer"],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)

print("=" * 70)
print("STDOUT")
print("=" * 70)
print(result.stdout)

print("=" * 70)
print("STDERR")
print("=" * 70)
print(result.stderr)

print("=" * 70)
print(f"Code retour : {result.returncode}")
print("=" * 70)

if result.returncode != 0:
    raise RuntimeError("❌ Tokenizer échoué — voir STDERR ci-dessus")

In [ ]:
import os
import subprocess

print("=" * 70)
print("ÉTAPE 12 — TOKENISATION DU DATASET")
print("=" * 70)

result = subprocess.run(
    ["poetry", "run", "python", "-m",
     "scripts.tokenize_dataset"],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)

print("=" * 70)
print("STDOUT")
print("=" * 70)
print(result.stdout)

print("=" * 70)
print("STDERR")
print("=" * 70)
print(result.stderr)

print("=" * 70)
print(f"Code retour : {result.returncode}")
print("=" * 70)

if result.returncode != 0:
    raise RuntimeError("❌ Tokenisation échouée — voir STDERR ci-dessus")

In [ ]:
import os
import subprocess

print("=" * 70)
print("ÉTAPE 13 — ENTRAÎNEMENT BITNET")
print("=" * 70)

result = subprocess.run(
    ["poetry", "run", "python", "-m",
     "scripts.bitnet.train_bitnet"],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)

print("=" * 70)
print("STDOUT")
print("=" * 70)
print(result.stdout)

print("=" * 70)
print("STDERR")
print("=" * 70)
print(result.stderr)

print("=" * 70)
print(f"Code retour : {result.returncode}")
print("=" * 70)

if result.returncode != 0:
    raise RuntimeError("❌ Entraînement échoué — voir STDERR ci-dessus")